# DSO 576 Module 6 — reproducible cleaning

National occupation-screening prototype. This notebook runs the reviewed pipeline from the original inputs; it is not a separate analysis. California funding conclusions require California data. No priority score, weights or thresholds.

Branch: https://github.com/emeraldo26/DSO576-Project-/tree/cleantaotianchen123  
Cleaning commit: `1f85f180ceb04b71e7bd921f706859c04317acbd` (the original cleaning submission, distinct from the later metadata commit).

Install `requirements-submission.txt`; run from the repository root or `code/`. The six hash-pinned sources must be present in `data/` (intentionally not committed). Every execution creates a fresh output directory; existing results and sources are not overwritten.

In [1]:
from pathlib import Path
import csv, hashlib, json, subprocess, sys, tempfile
import pandas as pd
from IPython.display import display
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'code/clean_module6.py').exists())
evidence = root / 'docs/module6_evidence'
readme = root / 'data/README.md'
readme_before = hashlib.sha256(readme.read_bytes()).hexdigest()
(root / 'outputs').mkdir(exist_ok=True)
work = Path(tempfile.mkdtemp(prefix='module6-notebook-', dir=root / 'outputs'))
run = work / 'clean'
def table(name):
    return pd.read_csv(run / name, dtype='string', keep_default_na=False)
print('Python:', sys.version.split()[0])
print('Original inputs only; fresh output directory; IDs read as text.')

Python: 3.11.4
Original inputs only; fresh output directory; IDs read as text.


In [2]:
result = subprocess.run([sys.executable, str(root / 'code/clean_module6.py'), '--output-dir', str(run)], cwd=root, capture_output=True, text=True, encoding='utf-8')
assert result.returncode == 0, result.stdout + result.stderr
manifest = json.loads((run / 'run_manifest.json').read_text())
assert manifest['checks_failed'] == 0 and manifest['checks_passed'] == 3529
print({k: manifest[k] for k in ['scope', 'checks_passed', 'checks_failed', 'priority_score']})
display(table('before_after_row_counts.csv'))

{'scope': 'National occupation-screening prototype', 'checks_passed': 3529, 'checks_failed': 0, 'priority_score': 'not calculated'}


,view,input_rows,output_rows,reduction,reason
0,penetration_lookup,17998,17991,7,Same normalized key and same value; retain all...
1,occupation_soc_bridge,1016,867,149,Use the unique .00 description for six-digit d...
2,soc_task_bridge,18838,18814,24,Consolidate identical task within one base SOC...
3,matched_task_links,18814,18656,158,Analytical penetration view requires a measure...
4,bls_line_items,1112,831,281,Exclude hierarchy summaries from detailed anal...
5,occupation_master,756,756,0,Left join retains every job occupation
6,bls_complete_analysis,756,755,1,Optional analysis requires BLS data; Legislato...
7,occupation_tasks,18656,17760,896,Task view for the 755 BLS-complete job occupat...
8,top_five_tasks,17760,3774,13986,"Top five display selection, penetration descen..."
9,jobs_clean,756,756,0,All original records and columns retained; par...


## Missing values, merges and denominators

Full audit files are retained in the fresh output directory. Source rows are all preserved; reductions apply only to named derived views. Blank strings, literal `None` and unavailable em dashes have different meanings.

In [3]:
missing = table('before_after_missing.csv')
display(missing)
display(table('merge_summary.csv'))
display(table('total_reconciliation.csv'))
display(table('metric_denominators.csv'))
records = table('five_record_checks.csv')
assert len(records) == 5 and records.match_status.eq('PASS').all()
display(records[['case', 'record_key', 'reason', 'expected', 'actual', 'match_status']])
print('All five original / expected / actual records are in five_record_checks.csv.')

,dataset,source_column,clean_column,before_blank,before_em_dash,before_literal_None,after_null,after_blank,after_em_dash,after_literal_None,rule
0,jobs,occ_code,occ_code,0,0,0,0,0,0,0,Preserve raw; typed companion; BLS categories ...
1,jobs,title,title,0,0,0,0,0,0,0,Preserve raw; typed companion; BLS categories ...
2,jobs,observed_exposure,observed_exposure_value,0,0,0,0,0,0,0,Preserve raw; typed companion; BLS categories ...
3,penetration,task,task,0,0,0,0,0,0,0,Preserve raw; typed companion; BLS categories ...
4,penetration,penetration,penetration_value,0,0,0,0,0,0,0,Preserve raw; typed companion; BLS categories ...
5,occupations,O*NET-SOC Code,O*NET-SOC Code,0,0,0,0,0,0,0,Preserve raw; typed companion; BLS categories ...
6,occupations,Title,Title,0,0,0,0,0,0,0,Preserve raw; typed companion; BLS categories ...
7,occupations,Description,Description,0,0,0,0,0,0,0,Preserve raw; typed companion; BLS categories ...
8,tasks,O*NET-SOC Code,O*NET-SOC Code,0,0,0,0,0,0,0,Preserve raw; typed companion; BLS categories ...
9,tasks,Title,Title,0,0,0,0,0,0,0,Preserve raw; typed companion; BLS categories ...


,relationship,status,keys
0,jobs_bls_line,both,755
1,jobs_bls_line,left_only,1
2,jobs_bls_line,right_only,76
3,jobs_onet_base,both,756
4,jobs_onet_base,right_only,111
5,penetration_onet_task,both,17429
6,penetration_onet_task,left_only,562
7,penetration_onet_task,right_only,149
8,task_fullcode_occupation,both,923
9,task_fullcode_occupation,right_only,93


,metric,published_total,line_items_831,matched_755,bls_only_76,line_minus_published,expected_line_minus_published,explanation
0,employment_2025_thousands,170280.8,170267.2,154223.4,16043.8,-13.6,-13.6,Snapshot control; rounding/excluded occupation...
1,employment_2035_thousands,176198.5,176184.5,158949.6,17234.9,-14,-14,Snapshot control; rounding/excluded occupation...
2,employment_change_thousands,5917.6,5917.9,4726.9,1191,0.299999999999,0.299999999999,Snapshot control; rounding/excluded occupation...
3,annual_openings_thousands,17461.8,17460.7,15607.2,1853.5,-1.1,-1.1,Snapshot control; rounding/excluded occupation...


,metric,records,nonmissing_denominator,numerator,mean,missing_policy
0,job_exposure_all,756,756,58.1944,0.0769767195767,exclude null from denominator; retain measured...
1,job_exposure_bls_complete,755,755,58.1944,0.0770786754967,exclude null from denominator; retain measured...
2,penetration_raw,17998,17998,1201.2546,0.0667437826425,exclude null from denominator; retain measured...
3,penetration_normalized_lookup,17991,17991,1196.1705,0.0664871602468,exclude null from denominator; retain measured...
4,employment_weighted_exposure,755,154223.4,19841.99913,0.128657513257,matched occupations only; denominator in thous...


,case,record_key,reason,expected,actual,match_status
0,1,00-0000,Preserve leading zeros and the national summar...,"{""Occupation type"": ""Summary"", ""employment_202...","{""Occupation type"": ""Summary"", ""employment_202...",PASS
1,2,11-1011,Recover the literal None category from the sam...,"{""ojt"": ""None"", ""ojt_csv_raw"": """", ""ojt_missin...","{""ojt"": ""None"", ""ojt_csv_raw"": """", ""ojt_missin...",PASS
2,3,51-2061,Keep independently rounded BLS published chang...,"{""annual_openings_thousands"": ""0.0"", ""employme...","{""annual_openings_thousands"": ""0"", ""employment...",PASS
3,4,11-1031,Left join retains Legislators and its measured...,"{""bls_match_status"": ""job_only"", ""employment_2...","{""bls_match_status"": ""job_only"", ""employment_2...",PASS
4,5,23-2011,Average only the seven matched tasks while exp...,"{""average_matched_task_penetration"": ""0.133857...","{""average_matched_task_penetration"": ""0.133857...",PASS


All five original / expected / actual records are in five_record_checks.csv.


## Independent check of the suggested BLS `None` recovery

Do not assume a CSV blank means no training. Independently read every coded workbook row with openpyxl, align by the original text code, compare literal string `None` (not Python null) to the CSV blank, and check the saved canonical value and exact workbook address. This cell does not import the cleaning module. It verifies the suggestion against local source cells, not external BLS provenance.

In [4]:
from openpyxl import load_workbook
with (root / 'data/bls_table_1_2_2025_35.csv').open(encoding='utf-8-sig', newline='') as f:
    raw = list(csv.DictReader(f))
code_col = '2025 National Employment Matrix code'
raw_by_code = {r[code_col]: r for r in raw}
wb = load_workbook(root / 'data/bls_occupation_tables_2025_35.xlsx', read_only=True, data_only=False)
ws = wb['Table 1.2']
clean = table('bls_table_1_2_clean.csv').set_index('soc_code')
audit = table('bls_category_recovery.csv')
headers = [cell.value for cell in ws[2]][:15]
counts = {'work_experience': 0, 'ojt': 0}
checked = []
seen = set()
for row in ws.iter_rows(min_row=3, max_row=1114, max_col=15):
    key = row[1].value
    assert isinstance(key, str) and key in raw_by_code and key not in seen
    seen.add(key)
    for col, canonical in [(13, 'work_experience'), (14, 'ojt')]:
        cell = row[col]
        if cell.value == 'None':
            assert isinstance(cell.value, str)
            assert raw_by_code[key][headers[col]] == ''
            assert clean.loc[key, canonical] == 'None'
            found = audit[(audit.soc_code == key) & (audit.column == headers[col])]
            assert len(found) == 1 and found.iloc[0].workbook_cell == cell.coordinate
            checked.append({'soc_code': key, 'column': canonical, 'cell': cell.coordinate, 'original_csv': '', 'original_workbook': cell.value, 'saved_clean': clean.loc[key, canonical], 'status': 'PASS'})
            counts[canonical] += 1
assert len(seen) == 1112 and counts == {'work_experience': 716, 'ojt': 321}
assert len(checked) == len(audit) == 1037
wb.close()
independent = {'status': 'PASS', 'coded_rows': len(seen), 'literal_None_recoveries': counts, 'cells_checked': len(checked), 'method': 'Independent openpyxl cell read; CSV code lookup; canonical saved CSV and recovery address comparison; no cleaning-module imports', 'external_provenance_verified': False}
print(json.dumps(independent, indent=2))
# This fresh check provides a compact, Git-eligible receipt for the submission PDF.
(evidence / 'independent_bls_none_check.json').write_text(json.dumps(independent, indent=2) + '\n', encoding='utf-8')
display(pd.DataFrame(checked).head(5))

{
  "status": "PASS",
  "coded_rows": 1112,
  "literal_None_recoveries": {
    "work_experience": 716,
    "ojt": 321
  },
  "cells_checked": 1037,
  "method": "Independent openpyxl cell read; CSV code lookup; canonical saved CSV and recovery address comparison; no cleaning-module imports",
  "external_provenance_verified": false
}


,soc_code,column,cell,original_csv,original_workbook,saved_clean,status
0,11-1011,ojt,O6,,None,None,PASS
1,11-1021,ojt,O7,,None,None,PASS
2,11-2011,ojt,O9,,None,None,PASS
3,11-2021,ojt,O11,,None,None,PASS
4,11-2022,ojt,O12,,None,None,PASS


In [5]:
report = work / 'reproducibility.json'
result = subprocess.run([sys.executable, str(root / 'code/verify_module6.py'), '--baseline', str(run), '--report', str(report)], cwd=root, capture_output=True, text=True, encoding='utf-8')
assert result.returncode == 0, result.stdout + result.stderr
repro = json.loads(report.read_text())
assert repro['status'] == 'PASS' and repro['files_compared'] == 41 and not repro['mismatches']
assert repro['submitted_sample_matches'] and repro['source_drift_rejected']
# Compare compact submitted evidence to this fresh run (JSON formatting is immaterial).
for saved in evidence.iterdir():
    fresh = run / saved.name
    if fresh.exists():
        if saved.suffix == '.json':
            assert json.loads(saved.read_text()) == json.loads(fresh.read_text()), saved.name
        else:
            assert saved.read_bytes() == fresh.read_bytes(), saved.name
assert hashlib.sha256(readme.read_bytes()).hexdigest() == readme_before
print('PASS: 41 byte-identical outputs; 50-row submitted sample identical; deliberate copied-source drift rejected.')
print('PASS: submitted compact evidence agrees with fresh results; data/README.md unchanged.')

PASS: 41 byte-identical outputs; 50-row submitted sample identical; deliberate copied-source drift rejected.
PASS: submitted compact evidence agrees with fresh results; data/README.md unchanged.


## Remaining limitations and handoff

Source preservation is verified, but original vendor release IDs / download dates and external BLS authenticity remain unresolved. Local workbook corroboration is not a verification of vendor provenance. Claude use is not all AI use or causal displacement. Conservative matching leaves tasks unknown; published rounded BLS values and wage populations must not be misinterpreted. The 50-row sample contains 48 master rows plus two explicitly labeled BLS controls and is not representative.

See `plan.md` section 8 for the exact checklist, `docs/module6_evidence/` for compact evidence and `code/README.md` for commands. All assertions above must pass; unexpected changes raise errors. No commit or push is performed.